In [1]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from shapely.geometry import Point, Polygon
import ipywidgets as widgets
from IPython.display import display, clear_output
import lonboard as lb
from shapely.ops import unary_union
import pandas as pd

Importing the road_specific_data_frames from GeodataFrame (.gpkg) of Denmark

In [2]:
# importing the road network data from the geopackage file
gdf_roadnetwork = gpd.read_file(r'C:\Users\vigne\Downloads\ES8-Project\simulator\GIS_data\GIS_mainroad_data\denmark.gpkg', layer="gis_osm_roads_free")

* Objective 1: Make a new GeodataFrame with only Secondary, Primary, and Tertiary roads
* Objective 2: Merge all the roads with the same name and different fclass into one row (if they have the same name, they are likely the same road split into multiple segments in the original data)

* Programming objective: 
    * Make a function def filtered_frame(gdf), The output must be a geodataframe with the above objectives fulfilled.

In [3]:
def filtered_frame(gdf):
    # Reproject everything to EPSG:4326 (WGS84) upfront for consistency.
    # lonboard also expects EPSG:4326, so this avoids downstream CRS issues.
    gdf = gdf.to_crs(epsg=4326)

    # Objective 1: Filter to only Secondary, Primary, and Tertiary roads
    target_fclasses = ["secondary", "primary", "tertiary"]
    gdf_filtered = gdf[gdf["fclass"].isin(target_fclasses)].copy()

    # Objective 2: Merge all roads with the same name into one row
    def merge_group(group):
        merged_geom = unary_union(group.geometry)
        row = group.iloc[0].copy()
        row['geometry'] = merged_geom
        row['fclass'] = ', '.join(sorted(group['fclass'].unique()))
        return row

    gdf_merged = (
        gdf_filtered[gdf_filtered["name"].notna() & (gdf_filtered["name"] != "")]
        .groupby("name", as_index=False)
        .apply(merge_group)
    )

    # The groupby result is already a GeoDataFrame with CRS from the input.
    # Use set_geometry to ensure geometry column is active, then reset index.
    gdf_merged = gpd.GeoDataFrame(gdf_merged, geometry='geometry').reset_index(drop=True)
    return gdf_merged


Function definition for plotting and interactive selection of road types

* Pipe Network Visualiser

(Note, declare an empty array which will serve as the selected_road_network and the target array to be updated)
1. Objective a: Develop a function which displays/ initialises the interactive map.
2. Objective b: Develop a click_handler function which select the road and appends the selected road on an empty array.
3. Objective c: Develop another click_handler linked to a widget which pops the previously inserted array object. 

In [15]:
selected_index = None
selected_roads = None

def show_map(gdf):
    global selected_index
    global selected_roads

    gdf_map = gdf[['fclass', 'name', 'osm_id', 'geometry']].copy()
    gdf_map = gdf_map.reset_index(drop=True)

    if gdf_map.crs is None:
        gdf_map = gdf_map.set_crs(epsg=4326)
    elif gdf_map.crs.to_epsg() != 4326:
        gdf_map = gdf_map.to_crs(epsg=4326)

    for col in gdf_map.columns:
        if col != 'geometry':
            gdf_map[col] = gdf_map[col].to_numpy().astype(object)

    road_layer = lb.PathLayer.from_geopandas(
        gdf_map,
        auto_downcast=False,
        get_color=[0, 128, 255],
        width_min_pixels=2,
        get_width=10,
        pickable=True,
    )

    output = widgets.Output()

    def on_selection_change(change):
        global selected_index
        global selected_roads

        idx = change["new"]
        if idx is None:
            return

        selected_index = idx
        osm_id = gdf_map.iloc[idx]["osm_id"]
        road_name = gdf_map.iloc[idx]["name"]

        with output:
            output.clear_output()

            # Deduplication: skip if osm_id already in selection
            if selected_roads is not None and not selected_roads.empty and osm_id in selected_roads["osm_id"].values:
                print(f"'{road_name}' (osm_id={osm_id}) already selected — skipping duplicate.")
                return

            # Use osm_id to locate the row in the original gdf
            new_selection = gdf[gdf["osm_id"] == osm_id]

            if selected_roads is None or selected_roads.empty:
                selected_roads = new_selection.copy()
            else:
                selected_roads = pd.concat([selected_roads, new_selection], ignore_index=True)

            print(f"Selected: '{road_name}' (lonboard idx={idx}, osm_id={osm_id})")
            print(f"Total accumulated roads: {len(selected_roads)}")
            display(selected_roads[['name', 'fclass', 'osm_id']].reset_index(drop=True))

    road_layer.observe(on_selection_change, names="selected_index")

    # Undo button: removes last appended road
    undo_btn = widgets.Button(description="Undo Last", button_style="warning")

    def on_undo(_):
        global selected_roads
        with output:
            output.clear_output()
            if selected_roads is not None and not selected_roads.empty:
                removed_name = selected_roads.iloc[-1]["name"]
                removed_osm_id = selected_roads.iloc[-1]["osm_id"]
                selected_roads = selected_roads.iloc[:-1].reset_index(drop=True)
                print(f"Removed: '{removed_name}' (osm_id={removed_osm_id}) | Remaining: {len(selected_roads)}")
                if not selected_roads.empty:
                    display(selected_roads[['name', 'fclass', 'osm_id']].reset_index(drop=True))
            else:
                print("No roads to remove.")

    undo_btn.on_click(on_undo)

    display(widgets.VBox([lb.Map([road_layer]), undo_btn, output]))


In [16]:
#Vizulaising the filtered data
gdf_filtered = filtered_frame(gdf_roadnetwork)
show_map(gdf_filtered_raw)

In [14]:
print(f"Total roads selected: {len(selected_roads)}")
selected_roads.head(len(selected_roads))

Total roads selected: 0


,name,osm_id,code,fclass,ref,oneway,maxspeed,layer,bridge,tunnel,geometry


Pipe generator Objectives
* HLR1 : Generate the 'Point' or 'MultiPoint' geometry at the ends of the road.
* HLR2 : Check if there are any intersections at any points of the road.
    *  LLR1: If yes, then generate a point directly at the intersection.
* HLR3 : Convert the road geometry (latitude&Longitude) to meters.
    *  LLR1: Divide the road length into parts of 12.5m
    *  LLR2: If the roads cannot be divided in equal parts of 12.5m then, the last road length must be more than 12.5m
* HLR4 : Save all these points in the format : 'Road Name': 'Node ID number' 'Intersection or Not' 'Names of intersecting Roads' 'Array of Neighbouring node IDs'
    *  LLR1: To get the neighbouring road ID.
    * *Implementation Note: Start with the first node ID, draw two radius of 12.5m and 25m. If the road lies between these two radii then we must register them as the neighbouring node (specifcally the same road). Additionally, if the node is an intersection then register the roads which are intersecting and use the previously mentioned radii identification method to register nodes which are in the neighbourhood only between the different roads* 

In [6]:
# HLR 1
from shapely.geometry import Point, MultiPoint, LineString, MultiLineString

def generate_endpoints(gdf):
    """
    Generates start and end points for each road.
    Returns a GeoDataFrame of endpoints.
    """

    endpoints = []

    for idx, row in gdf.iterrows():
        geom = row.geometry

        if isinstance(geom, LineString):
            start = Point(geom.coords[0])
            end = Point(geom.coords[-1])

            endpoints.append({
                "road_index": idx,
                "road_name": row.get("name"),
                "geometry": start,
                "position": "start"
            })

            endpoints.append({
                "road_index": idx,
                "road_name": row.get("name"),
                "geometry": end,
                "position": "end"
            })

        elif isinstance(geom, MultiLineString):
            for part in geom.geoms:
                start = Point(part.coords[0])
                end = Point(part.coords[-1])

                endpoints.append({
                    "road_index": idx,
                    "road_name": row.get("name"),
                    "geometry": start,
                    "position": "start"
                })

                endpoints.append({
                    "road_index": idx,
                    "road_name": row.get("name"),
                    "geometry": end,
                    "position": "end"
                })

    endpoints_gdf = gpd.GeoDataFrame(endpoints, crs=gdf.crs)

    return endpoints_gdf

In [256]:
# Exploded multilinestrings - selected roads may contain MultiLineStrings which can complicate topology handling.
def preprocess_roads(gdf):
    """
    Input:
        gdf (GeoDataFrame) — road network

    Output:
        GeoDataFrame — exploded LineStrings only

    Purpose:
        Convert MultiLineString into individual LineString rows.
        Makes topology handling easier.
    """

    gdf_clean = gdf.explode(index_parts=False).reset_index(drop=True)

    # Ensure geometry type is LineString
    gdf_clean = gdf_clean[gdf_clean.geometry.type == "LineString"]

    return gdf_clean


#=======================Debugging and Validation========================
exploded_gdf = preprocess_roads(selected_roads)
lb.viz(exploded_gdf)
#=======================End of Debugging and Validation=================

d:\Pipe_builder_gui\ml_env\Lib\site-packages\lonboard\_geoarrow\ops\reproject.py:40: UserWarning: No CRS exists on data. If no data is shown on the map, double check that your CRS is WGS84.
  warn(


In [10]:
# importing the road network data from the geopackage file
gdf_roadnetwork = gpd.read_file(r'C:\Users\vigne\Downloads\ES8-Project\simulator\GIS_data\GIS_mainroad_data\denmark.gpkg', layer="gis_osm_roads_free")

# Filter primary, secondary, and tertiary roads
target_fclasses = ["primary", "secondary", "tertiary"]
gdf_filtered_raw = gdf_roadnetwork[gdf_roadnetwork["fclass"].isin(target_fclasses)].copy()

# Save filtered roads as GeoJSON
gdf_filtered_raw.to_crs(epsg=4326).to_file(
    r'd:\Pipe_builder_gui\denmark_filtered_roads.geojson',
    driver="GeoJSON"
)

print(f"Total filtered roads: {len(gdf_filtered_raw)}")
print(gdf_filtered_raw["fclass"].value_counts())

Total filtered roads: 92750
fclass
tertiary     61996
secondary    20878
primary       9876
Name: count, dtype: int64


In [11]:
lb.viz(gdf_filtered_raw)

In [8]:
pwd

'd:\\Pipe_builder_gui'